# Week 7 — RFM Customer Segmentation for UrbanStyle
Team notebook: data cleaning → RFM scores → customer segments → charts for Marko.

In [ ]:
from pathlib import Path
import pandas as pd

data_candidates = [
    Path("sales_clean.csv"),
    Path("Week 7 - RFM Customer Segmentation/sales_clean.csv"),
]
data_path = next((candidate for candidate in data_candidates if candidate.is_file()), None)
if data_path is None:
    raise FileNotFoundError("Place the authorized sales_clean.csv beside this notebook or in its project folder.")

df = pd.read_csv(data_path)
print("Initial shape:", df.shape)


In [ ]:
dup_count = df.duplicated(subset=["invoice_id"]).sum()
print("Duplicate invoices:", dup_count)

df = df.drop_duplicates(subset=["invoice_id"], keep="first")
print("Shape after dedupe:", df.shape)


In [ ]:
print("NULLs per column:\n", df.isnull().sum())

df = df.dropna(subset=["customer_id", "sale_date", "total_price"])
print("Shape after NULL removal:", df.shape)


In [ ]:
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")

bad_dates = df["sale_date"].isnull().sum()
print("Unparseable dates:", bad_dates)

df = df.dropna(subset=["sale_date"])
print("Shape after date parsing:", df.shape)
print("sale_date dtype:", df["sale_date"].dtype)


In [ ]:
neg_count = (df["total_price"] <= 0).sum()
print("Non-positive total_price rows:", neg_count)

df = df[df["total_price"] > 0]
print("Shape after outlier removal:", df.shape)


In [ ]:
final_rows = df.shape[0]
unique_customers = df["customer_id"].nunique()
date_min = df["sale_date"].min()
date_max = df["sale_date"].max()

print("\n=== CLEANING REPORT ===")
print(f"Final shape: {final_rows} rows")
print(f"Unique customers: {unique_customers}")
print(f"Date range: {date_min.date()} → {date_max.date()}")
print(f"Duplicates removed: {dup_count}")
print("Critical NULLs remaining:",
      df[["customer_id", "sale_date", "total_price"]].isnull().sum().to_dict())
print("Non-positive total_price remaining:",
      (df["total_price"] <= 0).sum())


## RFM Analysis

In [ ]:
today = df['sale_date'].max() + pd.Timedelta(days=1)


In [ ]:
recency = (
    df.groupby("customer_id")["sale_date"]
      .max()
      .reset_index()
      .rename(columns={"sale_date": "last_purchase_date"})
)

recency["Recency"] = (today - recency["last_purchase_date"]).dt.days
recency.head()


In [ ]:
frequency = (
    df.groupby("customer_id")["sale_id"]
      .count()
      .reset_index()
      .rename(columns={"sale_id": "Frequency"})
)

frequency.head()


In [ ]:
monetary = (
    df.groupby("customer_id")["total_price"]
      .sum()
      .reset_index()
      .rename(columns={"total_price": "Monetary"})
)

monetary.head()


In [ ]:
rfm = recency.merge(frequency, on="customer_id") \
             .merge(monetary, on="customer_id")

rfm.head()


In [ ]:
# Recency score (lower recency = better)
rfm["R_score_raw"] = pd.qcut(rfm["Recency"], 5, labels=False) + 1
rfm["R_score"] = 6 - rfm["R_score_raw"]

# Frequency score (rank first to avoid ties)
rfm["Frequency_rank"] = rfm["Frequency"].rank(method="first")
rfm["F_score"] = pd.qcut(rfm["Frequency_rank"], 5, labels=False) + 1

# Monetary score
rfm["M_score"] = pd.qcut(rfm["Monetary"], 5, labels=False) + 1


In [ ]:
rfm["RFM_Score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

def assign_segment(score):
    if score >= 13:
        return "VIP Champions"
    elif score >= 10:
        return "Loyal"
    elif score >= 7:
        return "Potential"
    elif score >= 4:
        return "At Risk"
    else:
        return "Lost"

rfm["Segment"] = rfm["RFM_Score"].apply(assign_segment)

rfm.head()


In [ ]:
rfm["Segment"].value_counts()


# ROLE D — Visualization (RFM Insights)


In [ ]:
import plotly.express as px


In [ ]:
seg_counts = rfm["Segment"].value_counts().reset_index()
seg_counts.columns = ["Segment", "Count"]

fig1 = px.bar(
    seg_counts,
    x="Segment",
    y="Count",
    title="Customer Count by Segment",
    labels={"Segment": "Segment", "Count": "Customer Count"}
)

fig1.show()


In [ ]:
fig2 = px.scatter(
    rfm,
    x="Recency",
    y="Monetary",
    color="Segment",
    size="Frequency",
    title="Customer Segments (RFM Scatter Plot)",
    labels={
        "Recency": "Days Since Last Purchase",
        "Monetary": "Total Spend (EUR)",
        "Frequency": "Purchase Count"
    }
)

fig2.show()


In [ ]:
top_vips = (
    rfm[rfm["Segment"] == "VIP Champions"]
    .nlargest(10, "Monetary")
)
top_vips["customer_id"] = top_vips["customer_id"].astype(str)

fig3 = px.bar(
    top_vips,
    x="customer_id",
    y="Monetary",
    title="Top 10 VIP Customers by Total Spend",
    labels={"customer_id": "Customer ID", "Monetary": "Total Spend (EUR)"}
)

fig3.show()


In [ ]:
summary = rfm.groupby("Segment").agg(
    customers=("customer_id", "count"),
    revenue=("Monetary", "sum")
)
summary["% of customers"] = (summary["customers"] / summary["customers"].sum() * 100).round(1)
summary["% of revenue"] = (summary["revenue"] / summary["revenue"].sum() * 100).round(1)
summary.sort_values("revenue", ascending=False)

## Summary for Marko

**Key finding:** VIP Champions are 18% of customers but bring 43% of revenue. Together with Loyal customers, 45% of customers generate 73% of revenue.

**Urgent:** 529 customers (21%) are At Risk. They have spent €192k so far but have not bought recently.

**Recommendations:**
1. **VIP programme** – early access and exclusive offers to keep VIP Champions.
2. **Win-back campaign** – personalised emails with a discount for At Risk customers.
3. **Nurture Potential customers** – 759 customers who could grow into Loyal with follow-up offers.